# Zajęcia 1: klasyfikacja zgłoszeń helpdesk

Celem jest zbudowanie **prostego baseline’u regułowego**. Pracujemy wyłącznie na zbiorze `train`.

Wykonuj komórki kolejno. Nie używaj `expected_category`, `ticket_id`, numeru wiersza, `priority` ani `language` jako cech klasyfikatora. Wejściem są tylko `title` i `description`.

Po zajęciach notebook powinien zawierać:
- sprawdzony zbiór danych,
- co najmniej 10 sygnałów tekstowych,
- działającą funkcję `classify_ticket`,
- zaliczone testy kontraktu,
- accuracy na train,
- opis dwóch błędów i jedną hipotezę poprawki.


## 0. Importy i konfiguracja

Uruchom komórkę. Na pierwszych zajęciach wystarczą Python i `pandas`; nie korzystamy jeszcze z wykresów ani zaawansowanych metryk.

In [1]:
from pathlib import Path
import pandas as pd

ALLOWED_CATEGORIES = [
    "account-access", "network", "hardware",
    "software", "security", "other",
]
REQUIRED_COLUMNS = {
    "ticket_id", "title", "description",
    "expected_category", "priority", "language",
}

print("Konfiguracja: OK")


Konfiguracja: OK


## 1. Wczytanie danych

Przed uruchomieniem przewidź:
1. ile wierszy ma zbiór,
2. ile ma kolumn,
3. jakie trzy pierwsze kolumny zobaczysz.

Następnie uruchom komórkę i porównaj przewidywanie z wynikiem.

In [10]:
DATA_PATH = Path("tickets_train.csv")
assert DATA_PATH.exists(), (
    f"Nie znaleziono pliku {DATA_PATH.name}. "
    f"Umieść go obok notebooka. Bieżący katalog: {Path.cwd()}"
)

tickets = pd.read_csv(DATA_PATH)
print("Wymiary:", tickets.shape)
print("Kolumny:", tickets.columns.tolist())
display(tickets.head(3))


Wymiary: (200, 6)
Kolumny: ['ticket_id', 'title', 'description', 'expected_category', 'priority', 'language']


,ticket_id,title,description,expected_category,priority,language
0,TCK-0001,VPN disconnects,My VPN disconnects every few minutes when I wo...,network,normal,en
1,TCK-0002,Suspicious email,I received an email asking me to confirm my pa...,security,high,en
2,TCK-0003,Slow downloads over VPN,"When connected to VPN, downloads from SharePoi...",network,normal,en


## 2. Walidacja danych

Nie zmieniaj testów walidacyjnych tylko po to, aby „zniknął” błąd. Komunikat ma wskazać rzeczywisty problem w danych.

In [3]:
def validate_dataset(data: pd.DataFrame) -> None:
    missing_columns = REQUIRED_COLUMNS - set(data.columns)
    assert not missing_columns, f"Brak kolumn: {sorted(missing_columns)}"
    assert data.shape == (200, 6), f"Oczekiwano (200, 6), otrzymano {data.shape}"

    assert data["ticket_id"].notna().all(), "Brak ticket_id"
    assert data["ticket_id"].is_unique, "ticket_id nie są unikatowe"

    for column in ["title", "description", "expected_category", "priority", "language"]:
        assert data[column].notna().all(), f"Braki w kolumnie {column}"

    unknown = set(data["expected_category"]) - set(ALLOWED_CATEGORIES)
    assert not unknown, f"Nieznane kategorie: {sorted(unknown)}"

validate_dataset(tickets)
print("Walidacja danych: OK")


Walidacja danych: OK


## 3. Trzy fakty o zbiorze

Uruchom komórkę, a potem zapisz poniżej:
- rozmiar zbioru,
- najliczniejszą kategorię,
- najrzadszą kategorię lub kategorie.

**Moje trzy fakty:**
1. 200 wierszy, 6 kolumn
2. software (50 zgłoszeń)
3. security oraz other (obie po 20 zgłoszeń)


In [4]:
category_counts = tickets["expected_category"].value_counts()
print("Rozmiar zbioru:", tickets.shape)
print("Braki danych w całym zbiorze:", int(tickets.isna().sum().sum()))
print("Duplikaty całych wierszy:", int(tickets.duplicated().sum()))
display(category_counts.rename("liczba_zgloszen"))


Rozmiar zbioru: (200, 6)
Braki danych w całym zbiorze: 0
Duplikaty całych wierszy: 0


expected_category
software          50
account-access    44
network           36
hardware          30
security          20
other             20
Name: liczba_zgloszen, dtype: int64

## 4. Obejrzyj przykłady i zaprojektuj reguły

Wyświetl 20 przykładów. W materiale studenckim uzupełnij tabelę sygnałów. Szukaj zwrotów znaczących, a nie nazw kolumn ani identyfikatorów.

In [6]:
display(
    tickets[["title", "description", "expected_category"]]
    .head(20)
    .style.hide(axis="index")
)


title,description,expected_category
VPN disconnects,My VPN disconnects every few minutes when I work from home.,network
Suspicious email,I received an email asking me to confirm my password.,security
Slow downloads over VPN,"When connected to VPN, downloads from SharePoint are about 100 KB/s.",network
Nie działa podgląd PDF,"W Outlooku nie mogę podejrzeć załączników PDF, pojawia się komunikat o braku programu do podglądu.",software
Excel freezes with large files,Excel stops responding whenever I open the monthly sales report (about 80 MB). Smaller files work fine.,software
Bateria szybko się rozładowuje,"Bateria w laptopie trzyma tylko godzinę, wcześniej wytrzymywała cały dzień.",hardware
Wi-Fi lagguje na callach,"Na Wi-Fi w open space wszystko lagguje, zwłaszcza calle na Teamsach. Na kablu jest ok.",network
PowerPoint template missing,The new corporate PowerPoint template is not available under File > New. Colleagues have it.,software
Windows Hello PIN not available,My Windows Hello PIN stopped working and it says 'Something happened and your PIN isn't available'.,account-access
CRM app shows error 500,The Salesforce integration in our CRM client throws 'Internal Server Error 500' when I save a contact.,software


## 5. Baseline v0

Poniższy słownik jest punktem startowym. Rozszerz go tak, aby zawierał **co najmniej 10 sygnałów łącznie**. `other` pozostaje kategorią awaryjną, gdy nic nie pasuje.

Na pierwszych zajęciach kolejność kategorii rozstrzyga konflikt. To świadome uproszczenie, które później będzie można zweryfikować.

In [17]:
RULES = {
    "security": ["phishing", "suspicious email", "hacked", "malware"],
    "account-access": ["permission", "locked account", "password reset"],
    "network": ["vpn", "wi-fi", "connection lost", "router"],
    "hardware": ["battery", "charger", "broken screen"],
    "software": ["error", "install", "crash", "pdf", "podgląd"],
}

# TODO: dodaj własne sygnały. Razem powinno być ich co najmniej 10.

number_of_terms = sum(len(terms) for terms in RULES.values())
print("Liczba sygnałów:", number_of_terms)
assert number_of_terms >= 10, "Dodaj sygnały: wymagane jest co najmniej 10"


Liczba sygnałów: 19


In [18]:
def normalize_text(text) -> str:
    """Zamienia brak tekstu na pusty napis, usuwa nadmiar spacji i zmienia litery na małe."""
    if text is None:
        return ""
    return " ".join(str(text).lower().split())


def classify_ticket(title: str, description: str) -> dict:
    """Zwraca kategorię, umowną pewność i informację o potrzebie weryfikacji."""
    text = normalize_text(f"{normalize_text(title)} {normalize_text(description)}")

    for category, terms in RULES.items():
        if any(normalize_text(term) in text for term in terms):
            return {
                "category": category,
                "confidence": 0.7,
                "needs_human_review": False,
            }

    return {
        "category": "other",
        "confidence": 0.0,
        "needs_human_review": True,
    }

print("Funkcja klasyfikująca: gotowa")


Funkcja klasyfikująca: gotowa


## 6. Sprawdź trzy przypadki otwierające

Dla każdego przypadku porównaj wynik z decyzją podjętą na początku zajęć. Jeśli wynik jest inny, wskaż słowo lub brakujące słowo, które o tym zdecydowało.

In [19]:
opening_cases = [
    ("VPN disconnects", "My VPN disconnects every few minutes when I work from home."),
    ("Suspicious email", "I received an email asking me to confirm my password."),
    ("Windows Hello PIN", "My PIN stopped working and Windows says it is not available."),
]

for title, description in opening_cases:
    print(title, "->", classify_ticket(title, description))


VPN disconnects -> {'category': 'network', 'confidence': 0.7, 'needs_human_review': False}
Suspicious email -> {'category': 'security', 'confidence': 0.7, 'needs_human_review': False}
Windows Hello PIN -> {'category': 'other', 'confidence': 0.0, 'needs_human_review': True}


## 7. Test kontraktu

Test kontraktu sprawdza format wyniku, ale nie gwarantuje dobrej klasyfikacji. Jeśli test nie przechodzi, popraw funkcję, a nie test.

In [20]:
def run_contract_tests(classifier) -> None:
    cases = [
        ("", ""),
        ("Example", "Simple description"),
        ("ZAŻÓŁĆ", "mixed CASE 123"),
        (None, None),
    ]

    for title, description in cases:
        result = classifier(title, description)
        assert isinstance(result, dict)
        assert set(result) == {"category", "confidence", "needs_human_review"}
        assert result["category"] in ALLOWED_CATEGORIES
        assert isinstance(result["confidence"], (int, float))
        assert 0.0 <= result["confidence"] <= 1.0
        assert isinstance(result["needs_human_review"], bool)

    print("Testy kontraktu: OK")

run_contract_tests(classify_ticket)


Testy kontraktu: OK


## 8. Uruchomienie na całym zbiorze train

Ta komórka doda predykcję do kopii danych. Nie zmienia oryginalnego zbioru `tickets`.

In [21]:
def evaluate(data: pd.DataFrame, classifier) -> pd.DataFrame:
    outputs = data.apply(
        lambda row: classifier(row["title"], row["description"]),
        axis=1,
    )

    result = data.copy()
    result["predicted_category"] = [item["category"] for item in outputs]
    result["confidence"] = [item["confidence"] for item in outputs]
    result["needs_human_review"] = [item["needs_human_review"] for item in outputs]
    return result

results = evaluate(tickets, classify_ticket)
display(results.head(5))


,ticket_id,title,description,expected_category,priority,language,predicted_category,confidence,needs_human_review
0,TCK-0001,VPN disconnects,My VPN disconnects every few minutes when I wo...,network,normal,en,network,0.7,False
1,TCK-0002,Suspicious email,I received an email asking me to confirm my pa...,security,high,en,security,0.7,False
2,TCK-0003,Slow downloads over VPN,"When connected to VPN, downloads from SharePoi...",network,normal,en,network,0.7,False
3,TCK-0004,Nie działa podgląd PDF,"W Outlooku nie mogę podejrzeć załączników PDF,...",software,low,pl,software,0.7,False
4,TCK-0005,Excel freezes with large files,Excel stops responding whenever I open the mon...,software,normal,en,other,0.0,True


## 9. Pierwszy pomiar: accuracy na train

Accuracy to odsetek poprawnych odpowiedzi. Dzisiejszy wynik jest **roboczym pomiarem na train**, nie oceną działania na nowych danych.

In [22]:
results["is_correct"] = (
    results["expected_category"] == results["predicted_category"]
)
train_accuracy = results["is_correct"].mean()

print(f"Accuracy na train: {train_accuracy:.3f} ({train_accuracy:.1%})")
print("To nie jest jeszcze pomiar generalizacji.")


Accuracy na train: 0.315 (31.5%)
To nie jest jeszcze pomiar generalizacji.


## 10. Przeanalizuj dwa błędy

Uruchom komórkę i wybierz dwa wiersze. W materiale studenckim zapisz dla każdego:
- prawdziwą kategorię,
- predykcję,
- przyczynę błędu,
- hipotezę małej poprawki.


In [23]:
error_columns = [
    "ticket_id", "title", "description",
    "expected_category", "predicted_category",
    "confidence", "needs_human_review",
]
errors = results.loc[~results["is_correct"], error_columns]

print("Liczba błędów:", len(errors))
display(errors.head(10).style.hide(axis="index"))


Liczba błędów: 137


ticket_id,title,description,expected_category,predicted_category,confidence,needs_human_review
TCK-0005,Excel freezes with large files,Excel stops responding whenever I open the monthly sales report (about 80 MB). Smaller files work fine.,software,other,0.000000,True
TCK-0006,Bateria szybko się rozładowuje,"Bateria w laptopie trzyma tylko godzinę, wcześniej wytrzymywała cały dzień.",hardware,other,0.000000,True
TCK-0008,PowerPoint template missing,The new corporate PowerPoint template is not available under File > New. Colleagues have it.,software,other,0.000000,True
TCK-0009,Windows Hello PIN not available,My Windows Hello PIN stopped working and it says 'Something happened and your PIN isn't available'.,account-access,other,0.000000,True
TCK-0013,Teams screen sharing not working,"When I try to share my screen in Teams, the other participants only see a black window.",software,other,0.000000,True
TCK-0015,Email signature setup,How do I set up the new company email signature in Outlook? The generator link from the newsletter doesn't work.,software,other,0.000000,True
TCK-0016,Salary file sent to wrong recipient,I accidentally sent a file with employee salaries to an external email address.,security,other,0.000000,True
TCK-0017,Password posted in Teams chat,I pasted my password into a Teams chat with the whole team by mistake.,security,other,0.000000,True
TCK-0018,Zoom audio not working,"Zoom doesn't detect any microphone, although the same microphone works in Teams on this laptop.",software,other,0.000000,True
TCK-0020,Prośba o instalację programu,"Proszę o zainstalowanie programu 7-Zip na moim komputerze, nie mam uprawnień administratora.",software,other,0.000000,True


## 11. Jedna mała poprawka i zapis baseline’u

1. Wybierz **jedną** hipotezę z analizy błędów.
2. Zmień `RULES`.
3. Uruchom ponownie komórki od sekcji 5 do 10.
4. Sprawdź, czy test kontraktu nadal przechodzi.
5. Zapisz końcowy pomiar poniżej.

Nie poprawiaj rozwiązania wielokrotnie po obejrzeniu wszystkich odpowiedzi. Na kolejnych zajęciach użyjemy osobnego zbioru validation.


In [24]:
baseline_summary = {
    "version": "baseline-v0",
    "dataset": "train",
    "accuracy": round(float(train_accuracy), 3),
    "number_of_terms": sum(len(terms) for terms in RULES.values()),
    "analysed_errors": 2,
    "change_made": "Dodano słowa kluczowe 'pdf' oraz 'podgląd' do kategorii software.",
    "limitations": "Wynik policzony na train; reguły mogą nie uogólniać się na nowe dane.",
}

baseline_summary


{'version': 'baseline-v0',
 'dataset': 'train',
 'accuracy': 0.315,
 'number_of_terms': 19,
 'analysed_errors': 2,
 'change_made': "Dodano słowa kluczowe 'pdf' oraz 'podgląd' do kategorii software.",
 'limitations': 'Wynik policzony na train; reguły mogą nie uogólniać się na nowe dane.'}

## 12. Exit ticket

Odpowiedz samodzielnie:

1. Dlaczego `expected_category` nie może być cechą?
2. Czym różni się test kontraktu od pomiaru jakości?
3. Podaj jedną własną regułę i przypadek, który może ją zepsuć.
4. Dlaczego wynik na train nie wystarcza, aby uznać system za dobry?

**Odpowiedzi:**

1. Ponieważ jest to etykieta (wynik, który mamy przewidzieć), a nie cecha wejściowa. Jej użycie doprowadziłoby do wycieku danych (data leakage), ponieważ model znałby odpowiedź z góry.
2. Test kontraktu sprawdza poprawność techniczną (czy funkcja zwraca odpowiednie typy danych, klucze i dozwolone kategorie), natomiast pomiar jakości (accuracy) ocenia merytoryczną poprawność predykcji modelu względem rzeczywistych etykiet.
3. Reguła szukająca słowa "error" dla kategorii software. Może ją zepsuć zgłoszenie dotyczące błędu sieciowego („network error”), które zostanie błędnie zaklasyfikowane do oprogramowania zamiast do sieci.
4. Ponieważ model mógł zbytnio dopasować się do danych treningowych (overfitting) i na nowych, nieznanych wcześniej danych (np. zbiorze walidacyjnym) radzić sobie znacznie gorzej.
